In [ ]:
# Cell 0: API keys / credentials
# The ENTSOG Transparency Platform does not require an API key for the
# public operationalData endpoints used in this notebook. This cell is
# kept as a placeholder for consistency with other notebooks in this
# toolkit and in case ENTSOG introduces authenticated endpoints later.
# Never commit a real key here — keep this a placeholder in git history.
ENTSOG_API_KEY = None

# 01 — ENTSOG Gas Flow Ingestion

Fetches daily physical gas flow data from the ENTSOG Transparency
Platform for Germany, France, the Netherlands, Belgium, and Italy, and
exports it to parquet for downstream analysis.

In [ ]:
import sys
from datetime import date, timedelta
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent / "data"))

from entsog_client import EntsogClient, COUNTRIES, resolve_analysis_date

In [ ]:
# ANALYSIS_DATE = None auto-resolves to the latest available ENTSOG data point.
# Set an explicit date(YYYY, M, D) to pin the run to a specific end date instead.
ANALYSIS_DATE = None

LOOKBACK_DAYS = 365

client = EntsogClient()
resolved_date = resolve_analysis_date(ANALYSIS_DATE, client=client)
start_date = resolved_date - timedelta(days=LOOKBACK_DAYS)

print(f"ANALYSIS_DATE resolved to: {resolved_date}")
print(f"Fetching {start_date} .. {resolved_date} for countries: {COUNTRIES}")

In [ ]:
flows_df = client.get_physical_flows_multi(COUNTRIES, start_date, resolved_date)
flows_df.shape

In [ ]:
flows_df.head()

In [ ]:
flows_df.groupby("queryCountry").size().rename("records")

In [ ]:
output_dir = Path.cwd().parent / "data"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / f"entsog_flows_{resolved_date.isoformat()}.parquet"

flows_df.to_parquet(output_path, index=False)
print(f"Wrote {len(flows_df)} rows to {output_path}")